In [88]:
import os
import json
import netCDF4
import datetime
import calendar
import numpy as np

In [70]:
# Opening JSON file
f = open("/home/tang/Documents/noresm-land-sites-platform/resources/SVMC/data/events.json")
 
# returns JSON object as 
# a dictionary
data = json.load(f)

# order management events according to date
ordered_data = sorted(data['management']['events'], key = lambda x:datetime.datetime.strptime(x['date'], '%Y-%m-%d'), reverse=False)

f.close()

In [ ]:
# Write management in netcdf format

# variables to be created: 
#    - management_type: no management(0), harvest(1), fertilization(2), grazing(3)
#    - management specific variables: 
#      1 - harvest_biomass_output, harvest_c_output, (harvest_yield_harvest_dw_total, yield_C_at_harvest_total)
#      2 - fert_c_input, fert_n_input
#      3 - grazing_c_output, grazing_n_output (grazing_c_input and grazing_n_input can be reflected in fert_c_input & fert_n_input)

In [92]:
# Open the surface data file to modify

year=2021

os.remove("/home/tang/Documents/noresm-land-sites-platform/resources/SVMC/data/management_qvidja_2021.nc")
nc = netCDF4.Dataset('/home/tang/Documents/noresm-land-sites-platform/resources/SVMC/data/management_qvidja_2021.nc', 'w')

# Global attributes
nc.title = 'Management events - Qvidja'
nc.summary = ('All information related to management')
nc.keywords = 'Crop management'
nc.license = ('This work is licensed under a Creative Commons '
              'Attribution 4.0 International License.')
nc.references = ('FiON')
nc.source = 'events.json file on FMI-PECAN'
nc.Conventions = 'CF-1.6'
nc.institution = 'Finnish Meteorological Institute (FMI)'
nc.history = '{0} creation of Management events - Qvidja netcdf file.'.format(
              datetime.datetime.now().strftime("%Y-%m-%d")
             )

# -- Create dimensions
lat_dim = nc.createDimension('latitude', 1)
lon_dim = nc.createDimension('longitude', 1)
tim_dim = nc.createDimension('time', 365 + calendar.isleap(year))

# Create variables
lat_var = nc.createVariable('latitude', np.float64, ('latitude'))
lat_var.units = 'degrees_north'
lat_var.standard_name = 'latitude'
#lat_var.axis = 'Y'

lon_var = nc.createVariable('longitude', np.float64, ('longitude'))
lon_var.units = 'degrees_east'
lon_var.standard_name = 'longitude'
#lon_var.axis = 'X'

time_var = nc.createVariable('time', np.float64, ('time'))
time_var.standard_name = 'time'
time_var.calendar = 'gregorian'
#time_var.time_step = 'Monthly'
time_var.units = 'Days since 2021-01-01 00:00:00'
#time_var.axis = 'T'

# -- Create management variables:
mantype_var = nc.createVariable('management_type', np.int16, ('time', 'latitude', 'longitude'),
                           fill_value=9999)
mantype_var.units      = '1'
mantype_var.long_name  = 'Management types: 0. no management; 1. harvesting; 2. fertilization; 3. grazing.'
mantype_var.short_name = 'management_type'

manc_i_var = nc.createVariable('management_c_input', np.float64, ('time', 'latitude', 'longitude'),
                           fill_value=9999)
manc_i_var.units      = 'kg C m-2 s-1'
manc_i_var.long_name  = 'Carbon input to soil by management, e.g., fertilization (either inorganic or organic).'
manc_i_var.short_name = 'manangement_carbon_input'

mann_i_var = nc.createVariable('management_n_input', np.float64, ('time', 'latitude', 'longitude'),
                           fill_value=9999)
mann_i_var.units      = 'kg N m-2 s-1'
mann_i_var.long_name  = 'Nitrogen input to soil by management, e.g., fertilization (either inorganic or organic).'
mann_i_var.short_name = 'management_nitrogen_input'

manc_o_var = nc.createVariable('management_c_output', np.float64, ('time', 'latitude', 'longitude'),
                           fill_value=9999)
manc_o_var.units      = 'kg C m-2 s-1'
manc_o_var.long_name  = 'Carbon output from ecosystem due to management, e.g., animal grazing, harvest'
manc_o_var.short_name = 'management_carbon_output'

mann_o_var = nc.createVariable('management_n_output', np.float64, ('time', 'latitude', 'longitude'),
                           fill_value=9999)
mann_o_var.units      = 'kg N m-2 s-1'
mann_o_var.long_name  = 'Nitrogen output from ecosystem due to management, e.g., animal grazing, harvest.'
mann_o_var.short_name = 'management_nitrogen_output'

# -- Load values: time
#date_200506 = int((datetime.datetime(2005,6,1) - datetime.datetime(1970,1,1)).total_seconds())
time_values = np.arange(0.5,365+0.5+calendar.isleap(year),1.0)
time_var[:] = time_values 

# -- Load values: latitude and longitude
lat_values = [60.2942641727575]
lon_values = [22.3908939321246]
lat_var[:] = lat_values
lon_var[:] = lon_values

# -- Load values: management info, first create empty arrays of management variables
mantype_val = np.zeros(365+calendar.isleap(year), dtype=int)
manc_i_val  = np.zeros(365+calendar.isleap(year), dtype="float64") 
mann_i_val  = np.zeros(365+calendar.isleap(year), dtype="float64") 
manc_o_val  = np.zeros(365+calendar.isleap(year), dtype="float64") 
mann_o_val  = np.zeros(365+calendar.isleap(year), dtype="float64")

# -- Iterating through the json
for i in ordered_data:
  dt = datetime.datetime.strptime(i['date'], '%Y-%m-%d')
  if dt.year==2021:
    print(dt.year, dt.month, dt.day)
    #print(i)
    time_index=(dt - datetime.datetime(year,1,1)).days

    if i['mgmt_operations_event']=="harvest": 
      mantype_val[time_index] = 1
      manc_o_val[time_index] = i['yield_C_at_harvest_total']/10000/3600/24            # kg C/ha -> kg C /m2 /s-1 
      mann_o_val[time_index] = 0.0

    if i['mgmt_operations_event']=="fertilizer":
      #.or. i['mgmt_operations_event']=="organic_material" 
      mantype_val[time_index] = 2
      manc_i_val[time_index] = 0.0
      mann_i_val[time_index] = i['N_in_applied_fertilizer']/10000/3600/24            # kg N/ha -> kg N /m2 /s-1 

    if i['mgmt_operations_event']=="grazing": 
      dt_start        = datetime.datetime.strptime(i['grazing_period'][0], '%Y-%m-%d')
      dt_end          = datetime.datetime.strptime(i['grazing_period'][1], '%Y-%m-%d') 
      time_index_start= (dt_start - datetime.datetime(year,1,1)).days
      time_index_end  = (dt_end - datetime.datetime(year,1,1)).days

      mantype_val[time_index_start:time_index_end] = 3
      manc_i_val[time_index_start:time_index_end] = 170*0.5/10000/3600/24             # kg C/ha -> kg C /m2 /s-1 , from Laura
      mann_i_val[time_index_start:time_index_end] = 0.0
      manc_o_val[time_index_start:time_index_end] = 275*0.42/10000/3600/24            # kg N/ha -> kg N /m2 /s-1, from Laura 
      mann_o_val[time_index_start:time_index_end] = 0.0

# -- Load values: management info, (2) assign relevant management info to netcdf variables
mantype_var[:,0,0] = mantype_val[:]
manc_i_var[:,0,0]  = manc_i_val[:]
mann_i_var[:,0,0]  = mann_i_val[:] 
manc_o_var[:,0,0]  = manc_o_val[:]
mann_o_var[:,0,0]  = mann_o_val[:]

# Modify satellite phenology
#dset['MONTHLY_LAI'][:,:,:,:] = 0
#dset['MONTHLY_SAI'][:,:,:,:] = 0
#dset['MONTHLY_HEIGHT_TOP'][:,1,:,:] = 0
#dset['MONTHLY_HEIGHT_BOT'][:,1,:,:] = 0

# Modify topography
#dset['SLOPE'][:,:] = 0

nc.close()

2021 4 20
2021 4 27
2021 6 14
2021 8 30
2021 8 30


In [79]:
np.size(time_values)

364

In [64]:
datetime.datetime(2021,6,1).year

2021

In [66]:
time_values = np.arange(0.5,365-0.5+calendar.isleap(2021),1)

In [68]:
time_values[1]

1.5

In [85]:
manc_i_val  = np.zeros(365+calendar.isleap(year), dtype="float64") 

In [86]:
type(manc_i_val[0])

numpy.float32